<a href="https://colab.research.google.com/github/tamfatkh/measles/blob/main/G_Bayesian_Measles_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

your_path = 'yourpath/measles'
os.makedirs(your_path, exist_ok=True)
os.chdir(your_path)

In [ ]:
pip install pymc pytensor arviz matplotlib pandas

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor
import pytensor.tensor as pt
from pymc.ode import DifferentialEquation
from pytensor.compile.ops import as_op
from scipy.integrate import odeint
from scipy.optimize import least_squares
pytensor.config.floatX = "float32"

In [ ]:
import pandas as pd
data_path = "measles_TX.xlsx"
data_tx = pd.read_excel(data_path)
data_tx

In [ ]:
observed = data_tx[['I_0_4','I_5_17','I_18_plus']].values.astype("float32")
day_idx  = data_tx['day'].values.astype(int)
n_days   = int(data_tx['day'].max()) + 1
OBS_IS_DAILY = False

In [ ]:
C_TEXAS = np.array([
    [1.4101   ,  2.4226239 , 5.7905445 ],
    [0.8314527 , 9.311699  , 7.554599  ],
    [0.46766597 , 1.7777724 , 9.041952  ],
], dtype="float32")

N_tot_np = np.array([376327, 1096513, 4659604], dtype="float32")

C_init_np = np.array([0, 2, 0], dtype="float32")

In [ ]:
VE1 = 0.93
VE2 = 0.97

In [ ]:
def unpack_erlang_state(y, G, kE, kI):

    B = kE + kI + 3
    y = y.reshape(G, B)
    S = y[:, 0]
    E = y[:, 1:1+kE]
    I = y[:, 1+kE:1+kE+kI]
    R = y[:, 1+kE+kI]
    Ccum = y[:, 1+kE+kI+1]
    return S, E, I, R, Ccum

def pack_erlang_state(S, E, I, R, Ccum):
    return np.concatenate([S[:,None], E, I, R[:,None], Ccum[:,None]], axis=1).reshape(-1)

def erlang_rhs_numpy(y, C_t, q, fi, N, kE, kI, L_E, L_I, nu_t=None, VE=1.0):

    G = 3
    S, E, I, R, Ccum = unpack_erlang_state(y, G, kE, kI)
    Itot = I.sum(axis=1)
    lam = (q * C_t) @ (Itot / np.clip(N, 1e-12, np.inf))
    rE = (kE / L_E)
    rI = (kI / L_I)
    dS = -lam * S
    if nu_t is not None:
        dS -= (nu_t * VE) * S

    dE = np.zeros_like(E)
    dI = np.zeros_like(I)

    dE[:,0] = lam*S - rE*E[:,0]
    for j in range(1, kE):
        dE[:,j] = rE*(E[:,j-1] - E[:,j])

    dI[:,0] = rE*E[:,-1] - rI*I[:,0]
    for j in range(1, kI):
        dI[:,j] = rI*(I[:,j-1] - I[:,j])

    dR = rI*I[:,-1]
    if nu_t is not None:
        dR += (nu_t * VE) * S

    dC = fi * Itot

    return pack_erlang_state(dS, dE, dI, dR, dC)

def rk4_step_erlang_numpy(y, C_t, q, fi, N, kE, kI, L_E, L_I, nu_t=None, VE=1.0, dt=1.0):
    k1 = erlang_rhs_numpy(y, C_t, q, fi, N, kE, kI, L_E, L_I, nu_t, VE)
    k2 = erlang_rhs_numpy(y + 0.5*dt*k1, C_t, q, fi, N, kE, kI, L_E, L_I, nu_t, VE)
    k3 = erlang_rhs_numpy(y + 0.5*dt*k2, C_t, q, fi, N, kE, kI, L_E, L_I, nu_t, VE)
    k4 = erlang_rhs_numpy(y + dt*k3,     C_t, q, fi, N, kE, kI, L_E, L_I, nu_t, VE)
    return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def forward_erlang_numpy(q, frac_S, frac_E, frac_I, frac_R, fi,
                        N_tot_np, C_init_np, n_days,
                        C_schedule, kE, kI, L_E, L_I,
                        nu_schedule=None, VE=1.0):

    G=3
    S0 = N_tot_np*frac_S
    E0_tot = N_tot_np*frac_E
    I0_tot = N_tot_np*frac_I
    R0 = N_tot_np*frac_R
    C0 = C_init_np

    E0 = np.zeros((G,kE), dtype="float32") + (E0_tot[:,None]/kE).astype("float32")
    I0 = np.zeros((G,kI), dtype="float32") + (I0_tot[:,None]/kI).astype("float32")

    y = pack_erlang_state(S0, E0, I0, R0, C0)

    B = kE + kI + 3
    traj = np.zeros((n_days, G*B), dtype="float32")

    for t in range(n_days):
        C_t = C_schedule[t].astype("float32")
        nu_t = None if nu_schedule is None else nu_schedule[t].astype("float32")
        y = rk4_step_erlang_numpy(y, C_t, q, fi, N_tot_np, kE, kI, L_E, L_I, nu_t, VE, dt=1.0)
        traj[t] = y
    return traj

def extract_Ccum_from_traj(traj, kE, kI):

    G=3
    B = kE + kI + 3
    y = traj.reshape(-1, G, B)
    Ccum = y[:,:, -1]
    return Ccum.astype("float32")

def extract_S_from_traj(traj, kE, kI):
    G=3
    B = kE + kI + 3
    y = traj.reshape(-1, G, B)
    S = y[:,:, 0]
    return S.astype("float32")

In [ ]:
def erlang_rhs_pytensor(y, C_base, q, fi, N, kE, kI, L_E, L_I):

    G = 3
    B = kE + kI + 3
    Y = y.reshape((G, B))

    S = Y[:,0]
    E = Y[:,1:1+kE]
    I = Y[:,1+kE:1+kE+kI]
    R = Y[:,1+kE+kI]

    Itot = pt.sum(I, axis=1)
    lam = (q * C_base).dot(Itot / pt.clip(N, 1e-12, np.inf))

    rE = (kE / L_E)
    rI = (kI / L_I)

    dS = -lam * S


    dE0 = lam*S - rE*E[:,0]
    dE_list = [dE0]
    for j in range(1, kE):
        dE_list.append(rE*(E[:,j-1] - E[:,j]))
    dE = pt.stack(dE_list, axis=1)


    dI0 = rE*E[:,-1] - rI*I[:,0]
    dI_list = [dI0]
    for j in range(1, kI):
        dI_list.append(rI*(I[:,j-1] - I[:,j]))
    dI = pt.stack(dI_list, axis=1)

    dR = rI*I[:,-1]
    dC = fi * Itot

    dY = pt.concatenate(
        [
            dS[:,None],
            dE,
            dI,
            dR[:,None],
            dC[:,None],
        ],
        axis=1
    ).reshape((G*B,))
    return dY

def rk4_one_day_pytensor(y, C_base, q, fi, N, kE, kI, L_E, L_I):
    dt = pt.constant(np.float32(1.0))
    k1 = erlang_rhs_pytensor(y, C_base, q, fi, N, kE, kI, L_E, L_I)
    k2 = erlang_rhs_pytensor(y + 0.5*dt*k1, C_base, q, fi, N, kE, kI, L_E, L_I)
    k3 = erlang_rhs_pytensor(y + 0.5*dt*k2, C_base, q, fi, N, kE, kI, L_E, L_I)
    k4 = erlang_rhs_pytensor(y + dt*k3,     C_base, q, fi, N, kE, kI, L_E, L_I)
    return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def solve_erlang_scan(C_base, q, fi, y0, N, kE, kI, L_E, L_I, n_days):
    def one_step(y_prev, C_base, q, fi, N, L_E, L_I):
        return rk4_one_day_pytensor(y_prev, C_base, q, fi, N, kE, kI, L_E, L_I)
    outputs, _ = scan(
        fn=one_step,
        sequences=None,
        outputs_info=pt.as_tensor_variable(y0),
        non_sequences=[C_base, q, fi, N, L_E, L_I],
        n_steps=n_days
    )
    return outputs

In [ ]:
def build_model_fixed_k(kE, kI,
                        observed, day_idx, n_days,
                        C_base_np, N_tot_np, C_init_np,
                        OBS_IS_DAILY=False,
                        Rmax_by_age=None):

    G=3
    C_base = pt.as_tensor_variable(C_base_np.astype("float32"))
    N = pt.as_tensor_variable(N_tot_np.astype("float32"))

    if Rmax_by_age is None:
        Rmax_by_age = np.array([1, 1, 1], dtype="float32")
    Rmax_by_age = Rmax_by_age.astype("float32")

    Rmin = np.array([0.40, 0.40, 0.40], dtype="float32")

    with pm.Model() as model:
        log_q = pm.Normal("log_q", mu=np.log(0.05), sigma=1.0)
        q = pm.Deterministic("q", pm.math.exp(log_q))

        L_E = pm.TruncatedNormal("L_E", mu=12.0, sigma=3.0, lower=6.0, upper=25.0)
        L_I = pm.TruncatedNormal("L_I", mu=9.0, sigma=3.0, lower=4.0, upper=20.0)


        eps1 = pm.Beta("eps1", alpha=2.0, beta=2.0, shape=G)
        eps2 = pm.Beta("eps2", alpha=2.0, beta=2.0, shape=G)
        eps3 = pm.Beta("eps3", alpha=2.0, beta=2.0, shape=G)

        fi = pm.Beta("fi", alpha=2.0, beta=8.0, shape=G)

        R0_frac = pm.Deterministic(
            "frac_R",
            pt.as_tensor_variable(Rmin) + (pt.as_tensor_variable(Rmax_by_age) - pt.as_tensor_variable(Rmin)) * eps3
        )
        rem = 1.0 - R0_frac
        S0_frac = pm.Deterministic("frac_S", rem * eps1)
        rem2 = rem - S0_frac
        E0_frac = pm.Deterministic("frac_E", rem2 * eps2)
        I0_frac = pm.Deterministic("frac_I", rem2 * (1.0 - eps2))

        S0 = N * S0_frac
        E0tot = N * E0_frac
        I0tot = N * I0_frac
        R0 = N * R0_frac
        C0 = pt.as_tensor_variable(C_init_np.astype("float32"))

        E0 = pt.repeat((E0tot / kE)[:,None], kE, axis=1)
        I0 = pt.repeat((I0tot / kI)[:,None], kI, axis=1)

        y0 = pt.concatenate(
            [S0[:,None], E0, I0, R0[:,None], C0[:,None]],
            axis=1
        ).reshape((G*(kE+kI+3),))

        y_hat = solve_erlang_scan(C_base, q, fi, y0, N, kE, kI, L_E, L_I, n_days)

        B = kE + kI + 3
        Yobs = y_hat[day_idx].reshape((-1, G, B))
        C_hat = Yobs[:,:, -1]

        if OBS_IS_DAILY:
            mu_raw = pt.clip(pt.diff(C_hat, axis=0), 1e-6, np.inf)
            I_obs = pt.as_tensor_variable(observed[1:].astype("float32"))
        else:
            mu_raw = pt.clip(C_hat, 1e-6, np.inf)
            I_obs = pt.as_tensor_variable(observed.astype("float32"))

        mu_raw = pt.nan_to_num(mu_raw, nan=1e-6, posinf=1e6, neginf=1e-6)
        mu = pt.clip(mu_raw, 1e-6, 1e6).astype("float32")

        alpha = pm.HalfNormal("alpha", sigma=2.0)

        pm.NegativeBinomial("C_obs", mu=mu, alpha=pt.clip(alpha, 1e-6, np.inf), observed=pt.clip(I_obs, 0, np.inf))

        fracS0 = S0 / N
        K0 = (q*C_base) * fracS0[:,None]

        v = pt.ones((G,), dtype="float32")
        for _ in range(10):
            v = K0.dot(v); v = v/pt.sqrt(pt.sum(v**2))
        rho = pt.dot(v, K0.dot(v))/pt.dot(v,v)

        pm.Deterministic("Re0", rho * L_I)

    meta = dict(kE=kE, kI=kI, n_days=n_days, day_idx=day_idx,
                N_tot_np=N_tot_np, C_init_np=C_init_np, C_base_np=C_base_np)
    return model, meta

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

cov_KG_df = pd.read_excel("coverage_KG_2019_2024.xlsx")

cov_7th_df = pd.read_excel("coverage_7th_2019_2024.xlsx")

pop_csv_path = "Pop_Estimates_24.csv"

def _age_to_int(age_str: str) -> int:

    s = str(age_str).strip()
    s = s.replace("Year", "").replace("Years", "").strip()
    if s.startswith("<"):
        return 0

    if "+" in s:
        s = s.split("+", 1)[0].strip()

    digits = ""
    for ch in s:
        if ch.isdigit():
            digits += ch
        else:
            break
    if digits == "":
        raise ValueError(f"Could not parse age label: {age_str!r}")
    return int(digits)

def load_population_groups(pop_csv_path: str, year=2024) -> pd.DataFrame:

    df = pd.read_csv(pop_csv_path)

    df.columns = [c.strip() for c in df.columns]
    col_year = next((c for c in df.columns if c.lower() == "year"), None)
    col_cty  = next((c for c in df.columns if c.lower() == "county"), None)
    col_age  = next((c for c in df.columns if c.lower() == "age"), None)
    col_pop  = next((c for c in df.columns if "population" in c.lower()), None)

    if None in [col_year, col_cty, col_age, col_pop]:
        raise ValueError(f"Population CSV must contain Year/County/Age/Population columns. Got: {df.columns.tolist()}")

    df = df[df[col_year] == year].copy()

    df["age_int"] = df[col_age].apply(_age_to_int)
    df["pop"] = pd.to_numeric(df[col_pop], errors="coerce").fillna(0.0)

    def grp(a):
        if 0 <= a <= 4:
            return "0-4"
        if 5 <= a <= 17:
            return "5-17"
        return "18+"

    df["age_group"] = df["age_int"].apply(grp)

    out = (
        df.groupby([col_cty, "age_group"], as_index=False)["pop"]
          .sum()
          .pivot(index=col_cty, columns="age_group", values="pop")
          .fillna(0.0)
          .reset_index()
          .rename_axis(None, axis=1)
    )

    for g in ["0-4", "5-17", "18+"]:
        if g not in out.columns:
            out[g] = 0.0

    out = out.rename(columns={
        col_cty: "County",
        "0-4": "pop_0_4",
        "5-17": "pop_5_17",
        "18+": "pop_18p"
    })
    return out


def _to_frac(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip()
        if s.endswith("%"):
            s = s[:-1].strip()
            try:
                return float(s) / 100.0
            except:
                return np.nan
        try:
            v = float(s)
        except:
            return np.nan
        return v / 100.0 if v > 1.0 else v
    try:
        v = float(x)
    except:
        return np.nan
    return v / 100.0 if v > 1.0 else v

def _ensure_county_column(df_cov: pd.DataFrame) -> pd.DataFrame:

    df = df_cov.copy()
    df.columns = [str(c).strip() for c in df.columns]

    if "County" in df.columns:
        return df

    if df.index.name and str(df.index.name).strip().lower() == "county":
        df = df.reset_index().rename(columns={df.index.name: "County"})
        return df

    df = df.reset_index(drop=True)
    first = df.columns[0]
    df = df.rename(columns={first: "County"})
    return df

def summarize_coverage(df_cov: pd.DataFrame,
                       years=(2019, 2020, 2021, 2022, 2023, 2024),
                       method="gmean") -> pd.Series:

    df = _ensure_county_column(df_cov)

    df.columns = [str(c).strip() for c in df.columns]
    years = [str(y) for y in years]

    present = [y for y in years if y in df.columns]
    if len(present) == 0:
        raise ValueError(f"No requested year columns found. Wanted {years}, got {df.columns.tolist()}")

    for y in present:
        df[y] = df[y].apply(_to_frac)

    X = df[present].to_numpy(dtype=float)

    if method == "mean":
        vals = np.nanmean(X, axis=1)
    elif method == "min":
        vals = np.nanmin(X, axis=1)
    elif method == "gmean":
        vals = gmean(X, axis=1)
    elif method == "min":
        vals = np.nanmin(X, axis=1)
    elif method == "p10":
        vals = np.nanpercentile(X, 10, axis=1)
    else:
        raise ValueError("method must be one of: 'mean', 'min', 'p10'")

    out = pd.Series(vals, index=df["County"].astype(str).str.strip(), name=f"cov_{method}")
    out = out.clip(lower=0.0, upper=1.0)
    return out


def compute_Ra0_upper_bounds(pop_csv_path: str,
                             cov_KG_df: pd.DataFrame,
                             cov_7th_df: pd.DataFrame,
                             counties=None,
                             year_pop=2024,
                             years_cov=(2019, 2020, 2021, 2022, 2023, 2024),
                             method_cov="gmean",                   # "gmean"|"mean"|"min"|"p10"
                             VE0_4=0.93,
                             VE5_17=0.97,
                             adult_cap=0.99,
                             scale0_4=1.0,
                             cap_hardmax=0.99):

    pop_tab = load_population_groups(pop_csv_path, year=year_pop)

    if counties is not None:
        counties_set = set([str(c).strip() for c in counties])
        pop_tab = pop_tab[pop_tab["County"].astype(str).str.strip().isin(counties_set)].copy()

    covKG = summarize_coverage(cov_KG_df, years=years_cov, method=method_cov)
    cov7  = summarize_coverage(cov_7th_df, years=years_cov, method=method_cov)

    county_tab = pop_tab.copy()
    county_tab["County"] = county_tab["County"].astype(str).str.strip()
    county_tab["covKG"] = county_tab["County"].map(covKG).astype(float)
    county_tab["cov7"]  = county_tab["County"].map(cov7).astype(float)

    county_tab["cap0_4"]  = (county_tab["covKG"] * float(VE0_4) * float(scale0_4)).clip(0.0, cap_hardmax)
    county_tab["cap5_17"] = (county_tab["cov7"]  * float(VE5_17)).clip(0.0, cap_hardmax)
    county_tab["cap18p"]  = float(adult_cap)

    def wavg(val_col, w_col):
        v = county_tab[val_col].to_numpy(dtype=float)
        w = county_tab[w_col].to_numpy(dtype=float)
        m = np.isfinite(v) & np.isfinite(w) & (w > 0)
        if m.sum() == 0:
            return np.nan
        return float(np.sum(v[m] * w[m]) / np.sum(w[m]))

    region_caps = {
        "cap0_4":  wavg("cap0_4",  "pop_0_4"),
        "cap5_17": wavg("cap5_17", "pop_5_17"),
        "cap18p":  float(adult_cap),
        "N0_4":    float(county_tab["pop_0_4"].sum()),
        "N5_17":   float(county_tab["pop_5_17"].sum()),
        "N18p":    float(county_tab["pop_18p"].sum()),
        "method_cov": method_cov,
        "years_cov": tuple(int(y) for y in years_cov),
        "VE0_4": float(VE0_4),
        "VE5_17": float(VE5_17),
        "scale0_4": float(scale0_4),
    }

    return county_tab, region_caps


def region_Rmax_array(region_caps: dict) -> np.ndarray:
    return np.array([region_caps["cap0_4"], region_caps["cap5_17"], region_caps["cap18p"]], dtype=np.float32)

county_tab, region_caps = compute_Ra0_upper_bounds(
     pop_csv_path="Pop_Estimates_24.csv",
     cov_KG_df=cov_KG_df,
     cov_7th_df=cov_7th_df,
     counties=None,
     method_cov="gmean",              # "mean" or "min" or "p10"
     VE0_4=0.93,
     VE5_17=0.97,
     adult_cap=1.0,
     scale0_4=1.0
 )
Rmax_by_age = region_Rmax_array(region_caps)
print("Region caps:", region_caps)
print("Rmax_by_age:", Rmax_by_age)
display(county_tab)

In [ ]:
k_grid = [(1,1),(1,2),(2,1),(2,2)]
models = {}
priors = {}
traces = {}
loos = {}
results = []

for (kE, kI) in k_grid:
    model, meta = build_model_fixed_k(
        kE, kI,
        observed=observed, day_idx=day_idx, n_days=n_days,
        C_base_np=C_TEXAS, N_tot_np=N_tot_np, C_init_np=C_init_np,
        OBS_IS_DAILY=OBS_IS_DAILY,
        Rmax_by_age=Rmax_by_age
    )
    models[(kE,kI)] = (model, meta)

In [ ]:
DO_PRIOR = True
if DO_PRIOR:
    for (kE, kI), (model, meta) in models.items():
        with model:
            prior = pm.sample_prior_predictive(
                samples=1000,
                var_names=["q","L_E","L_I","fi","alpha","Re0","frac_S","frac_E","frac_I","frac_R"],
                random_seed=42,
                return_inferencedata=True
            )
        priors[(kE,kI)] = prior

In [ ]:
import numpy as np

def prior_draws_1d(idata, var, group="prior"):
    x = idata[group][var].values
    return np.ravel(x)

In [ ]:
import matplotlib.pyplot as plt

vars_to_plot = ["q","L_E","L_I","fi","alpha","Re0","frac_S","frac_E","frac_I","frac_R"]

def plot_prior_hists_for_model(idata, title_prefix="", bins=20):
    n = len(vars_to_plot)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 3.2*nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, v in zip(axes, vars_to_plot):
        s = prior_draws_1d(idata, v)
        ax.hist(s, bins=bins)
        ax.set_title(f"{title_prefix}{v}")
        ax.set_ylabel("count")
    for ax in axes[len(vars_to_plot):]:
        ax.axis("off")

    fig.tight_layout()
    return fig

for (kE, kI), idata in priors.items():
    plot_prior_hists_for_model(idata, title_prefix=f"(kE={kE}, kI={kI})  ")
plt.show()

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt


with model:
    prior_pb_kw = {}
    if "progressbar" in inspect.signature(pm.sample_prior_predictive).parameters:
        prior_pb_kw["progressbar"] = True

    prior_idata = pm.sample_prior_predictive(
        samples=1000,
        var_names=["C_obs"],
        return_inferencedata=True,
        random_seed=42,
        **prior_pb_kw
    )


Csim = prior_idata.prior_predictive["C_obs"].values

if Csim.ndim == 4:
    Csim = Csim.reshape((-1, Csim.shape[-2], Csim.shape[-1]))
elif Csim.ndim == 3:
    pass
else:
    raise ValueError(f"Unexpected C_obs shape: {Csim.shape}")

if OBS_IS_DAILY:
    t = data_tx["day"].values.astype(int)[1:]
    obs_plot = observed[1:].astype(float)
else:
    t = data_tx["day"].values.astype(int)
    obs_plot = observed.astype(float)

fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharex=True)
age_names = ["0–4", "5–17", "18+"]

for j in range(3):
    lo = np.percentile(Csim[:, :, j], 25, axis=0)
    md = np.percentile(Csim[:, :, j], 50, axis=0)
    hi = np.percentile(Csim[:, :, j], 75, axis=0)

    axs[j].scatter(t, obs_plot[:, j], s=20, color="k", alpha=0.6, label="Data")
    axs[j].fill_between(t, lo, hi, alpha=0.25, label="IQ-region")
    axs[j].plot(t, md, lw=1.5, label="Prior Median")
    axs[j].set_title(f"Age group {age_names[j]}")
    axs[j].set_xlabel("Day")
    axs[j].set_ylabel("Number of Cases")

axs[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
DO_POSTERIOR = True
if DO_POSTERIOR:
    for (kE, kI), (model, meta) in models.items():
        with model:
            idata = pm.sample(
                draws=1000, tune=1000, chains=5, cores=5, target_accept=0.95,
                random_seed=[2,3,4,5,6],
                idata_kwargs={"log_likelihood": True},
                progressbar=True
            )
            loo = az.loo(idata)

        traces[(kE,kI)] = idata
        loos[(kE,kI)] = loo
        results.append((kE,kI, float(loo.elpd_loo), float(loo.p_loo)))

best_kE, best_kI, best_elpd, best_p = max(results, key=lambda x: x[2])
print("Best (kE,kI):", (best_kE, best_kI), "elpd_loo:", best_elpd, "p_loo:", best_p)

trace = traces[(best_kE,best_kI)]
model, meta = models[(best_kE,best_kI)]

print(az.summary(trace, var_names=["q","L_E","L_I","fi","alpha","Re0"], round_to=2))

In [ ]:
az.plot_trace(trace)
plt.show()

In [ ]:
az.plot_autocorr(trace)
plt.show()

In [ ]:
az.plot_rank(trace)
plt.show()

In [ ]:
az.plot_ess(trace,kind = "evolution")
plt.show()

In [ ]:
with model:
    idata_ppc = pm.sample_posterior_predictive(trace, var_names=["C_obs"], random_seed=42, return_inferencedata=True)

ppc = idata_ppc.posterior_predictive["C_obs"].rename({"C_obs_dim_0":"day","C_obs_dim_1":"age"})
median = ppc.median(dim=("chain","draw")).to_numpy()
hdi = az.hdi(ppc, hdi_prob=0.95)["C_obs"]
hdi_low  = hdi.sel(hdi="lower").to_numpy()
hdi_high = hdi.sel(hdi="higher").to_numpy()
days = data_tx["day"].values.astype(int)
y_obs_raw = data_tx[["I_0_4", "I_5_17", "I_18_plus"]].values.astype("float32")

plot_days = days.copy()
obs_plot = y_obs_raw if not OBS_IS_DAILY else y_obs_raw[1:]

fig, axs = plt.subplots(1,3, figsize=(14,3), sharex=True)
labels = ["0–4","5–17","18+"]
for j, ax in enumerate(axs):
    ax.scatter(plot_days, obs_plot[:,j], s=18, color="k", alpha=0.6, label="Data")
    ax.plot(plot_days, median[:,j], lw=2, label="Posterior median")
    ax.fill_between(plot_days, hdi_low[:,j], hdi_high[:,j], alpha=0.25, label="95% HDI")
    ax.set_title(labels[j])
    ax.set_xlabel("Day")
axs[0].set_ylabel("Cases")
axs[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
best_kE, best_kI, best_elpd, best_p = max(results, key=lambda x: x[2])
print(best_kE, best_kI, best_elpd, best_p)

In [ ]:
az.plot_trace(trace, var_names=['q','frac_E','frac_I','frac_R','alpha','Re0'])
az.summary(trace, round_to=2)

In [ ]:
import arviz as az
import matplotlib.pyplot as plt

az.plot_trace(trace, var_names=['q','frac_E','frac_I','frac_R','alpha','Re0'])
plt.tight_layout()
plt.show()

az.plot_autocorr(trace, max_lag=50)
plt.tight_layout()
plt.show()

summary_all = az.summary(trace, round_to=7,
    stat_funcs={ "median": np.median })
print(summary_all)


In [ ]:
import numpy as np
axes = az.plot_trace(trace, var_names=['q','L_E','L_I','frac_E','frac_I','frac_R','alpha','Re0'])

axes = np.atleast_2d(np.asarray(axes))
pretty = {
    "q": r"Transmission Probability $q$",
    "L_E": "Mean Latent Period Length",
    "L_I": "Mean Infectious Period Length",
    "frac_E": "Initial E Fractions",
    "frac_I": "Initial I Fractions",
    "frac_R": "Initial R Fractions",
    "alpha": r"$\alpha$",
    "Re0": r"$R_e$",
}
vars_order = list(pretty)

for i, v in enumerate(vars_order):
    axes[i, 0].set_title(f"Posterior Distributions of {pretty[v]}")
    axes[i, 1].set_title(f" Traces of {pretty[v]}")

plt.tight_layout()
plt.show()

In [ ]:
ax = az.plot_posterior(
    trace,
    var_names=['frac_R'],
    coords={'frac_R_dim_0': [0]},
    hdi_prob=0.95
)

ax.set_title(" ")
ax.set_xlabel("$R_0(0)/N_0$", fontsize=14, fontname="Times New Roman")

plt.show()

In [ ]:
ax = az.plot_posterior(
    trace,
    var_names=['frac_R'],
    coords={'frac_R_dim_0': [1]},
    hdi_prob=0.95
)

ax.set_title(" ")
ax.set_xlabel("$R_1(0)/N_1$", fontsize=14, fontname="Times New Roman")

plt.show()

In [ ]:
ax = az.plot_posterior(
    trace,
    var_names=['frac_R'],
    coords={'frac_R_dim_0': [2]},
    hdi_prob=0.95
)

ax.set_title(" ")
ax.set_xlabel("$R_2(0)/N_2$", fontsize=14, fontname="Times New Roman")

plt.show()

In [ ]:
az.summary(
    trace,
    var_names=['Re0']
)

In [ ]:
import matplotlib.pyplot as plt
import arviz as az

axes = az.plot_posterior(
    trace,
    var_names=['Re0'],
    hdi_prob=0.95,
)

ax = axes if isinstance(axes, plt.Axes) else axes[0][0]

ax.set_title("")
ax.set_title(None)

ax.set_xlabel("$R_e$", fontsize=14, fontname="Times New Roman")

plt.show()

In [ ]:
observed_I_data = data_tx[['I_0_4', 'I_5_17', 'I_18_plus']].values
n_days = observed_I_data.shape[0]
t_eval = data_tx['day'].values
observed_I_data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

days            = data_tx['day'].values
observed_I_data = data_tx[['I_0_4','I_5_17','I_18_plus']].values
age_labels      = ['0–4','5–17','18+']

with model:
    idata_ppc = pm.sample_posterior_predictive(
        trace,
        var_names=["C_obs"],
        random_seed=42,
        return_inferencedata=True,
    )
ppc = idata_ppc.posterior_predictive["C_obs"]

ppc = ppc.rename({"C_obs_dim_0": "day", "C_obs_dim_1": "age"})

median = ppc.median(dim=("chain", "draw")).to_numpy()
hdi_ds = az.hdi(ppc, hdi_prob=0.95)
hdi_da = hdi_ds["C_obs"]
hdi_low  = hdi_da.sel(hdi="lower").to_numpy()
hdi_high = hdi_da.sel(hdi="higher").to_numpy()

fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for i, ax in enumerate(axs):
    obs = observed_I_data[:, i] if isinstance(observed_I_data, np.ndarray) \
          else observed_I_data.iloc[:, i].values

    ax.scatter(days, obs, s=20, color="k", alpha=0.6, label="Data")
    ax.plot(days, median[:, i], lw=1.5, label="Posterior Median")
    ax.fill_between(days,
                    hdi_low[:, i],
                    hdi_high[:, i],
                    alpha=0.3,
                    label="95% HDI")
    ax.set_ylabel("Number of Cases")
    ax.set_title(f"Age group {age_labels[i]}")
    ax.legend(frameon=True)
    ax.grid(False)

axs[-1].set_xlabel("Day")
plt.tight_layout()
plt.show()